# OFFBALL — Regresión del valor de mercado

**Objetivo:** estimar el valor de mercado de un futbolista (en euros) usando su edad, posición y estadísticas de rendimiento.

Este notebook hace la **preparación de datos**, construye `df2_regression`, compara modelos y evalúa una configuración seleccionada sobre valoraciones futuras. Para que las ejecuciones posteriores sean rápidas, guarda un caché de la preparación en Drive y lo reutiliza mientras los CSV limpios no cambien.

- No usamos PySpark.
- Los CSV limpios originales no se modifican.
- Solo usamos estadísticas de temporadas terminadas **antes** de cada valoración.
- **Train candidatos:** hasta 2022; **Validation:** 2023; **Test final:** 2024 en adelante.
- Validation sirve para elegir configuración. Test se reserva para la evaluación final.
- MLflow no se instala ni se arranca como servidor desde este notebook. Si la librería ya está disponible, se usa para registrar experimentos y modelos en un tracking local. Si no, el entrenamiento sigue y guarda los artefactos localmente.
- Las variables de clustering/GMM se incorporarán más adelante, cuando estén preparadas.

Ejecuta las celdas en orden. Cada sección explica qué hacemos y por qué.

## 1. Cargar datos y preparar rutas

Montamos Google Drive, definimos las carpetas del proyecto y cargamos solo las columnas que necesitamos para reducir el uso de memoria.

In [ ]:
# ==========================================================
# 1. CARGA DE DATOS, RUTAS Y CACHÉ
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Carpeta principal del proyecto en Drive
OFFBALL_DIR = Path('/content/drive/MyDrive/PROYECTO-IGOR-CARMELO-BYRON/OFFBALL')
CLEAN_DIR = OFFBALL_DIR / 'dataset_limpio'
OUTPUT_DIR = OFFBALL_DIR / 'Machine_Learning' / 'DF2_Regresion'
MODELS_DIR = OUTPUT_DIR / 'modelos'
MLFLOW_DIR = OUTPUT_DIR / 'mlruns'

for carpeta in [OUTPUT_DIR, MODELS_DIR, MLFLOW_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)


archivos_necesarios = ['players.csv', 'appearances.csv', 'games.csv', 'player_valuations.csv']
rutas_fuente = [CLEAN_DIR / archivo for archivo in archivos_necesarios]
for ruta in rutas_fuente:
    if not ruta.exists():
        raise FileNotFoundError(f'No se encuentra el archivo: {ruta}')

# El caché se invalida automáticamente si cambia el tamaño o fecha de alguno de los CSV limpios.
CACHE_DF2 = OUTPUT_DIR / 'cache_df2_regression.csv'
CACHE_AUDITORIA = OUTPUT_DIR / 'cache_auditoria_filtros.csv'
CACHE_COBERTURA = OUTPUT_DIR / 'cache_cobertura_estadisticas.csv'
CACHE_METADATA = OUTPUT_DIR / 'cache_metadata.json'
FIRMA_FUENTES = {
    ruta.name: {'size': ruta.stat().st_size, 'mtime_ns': ruta.stat().st_mtime_ns}
    for ruta in rutas_fuente
}
CACHE_CARGADO = False

if all(ruta.exists() for ruta in [CACHE_DF2, CACHE_AUDITORIA, CACHE_COBERTURA, CACHE_METADATA]):
    try:
        with open(CACHE_METADATA, 'r', encoding='utf-8') as f:
            metadata_cache = json.load(f)
        if metadata_cache.get('firma_fuentes') == FIRMA_FUENTES:
            df2_regression = pd.read_csv(CACHE_DF2, parse_dates=['valuation_date', 'latest_stats_end'])
            df_auditoria_filtros = pd.read_csv(CACHE_AUDITORIA)
            df_cobertura_anual = pd.read_csv(CACHE_COBERTURA)
            CACHE_CARGADO = True
            print('CACHÉ VÁLIDO: dataset de regresión cargado desde Drive.')
            print('Se saltan las uniones y agregaciones grandes de las secciones 2–7.')
        else:
            print('Caché antiguo: los CSV limpios han cambiado. Se recalculará el dataset.')
    except Exception as error:
        print('No se pudo leer el caché; se recalculará:', type(error).__name__)

if not CACHE_CARGADO:
    print('Primera ejecución o caché inválido: cargando CSV limpios y preparando datos...')
    df_players = pd.read_csv(
        CLEAN_DIR / 'players.csv',
        usecols=['player_id', 'date_of_birth', 'position', 'sub_position']
    )
    df_appearances = pd.read_csv(
        CLEAN_DIR / 'appearances.csv',
        usecols=['appearance_id', 'game_id', 'player_id', 'competition_id', 'minutes_played', 'goals', 'assists']
    )
    df_games = pd.read_csv(
        CLEAN_DIR / 'games.csv',
        usecols=['game_id', 'competition_id', 'season', 'date'],
        parse_dates=['date']
    )
    df_valuations = pd.read_csv(
        CLEAN_DIR / 'player_valuations.csv',
        usecols=['player_id', 'date', 'market_value_in_eur'],
        parse_dates=['date']
    )
    df_valuations['market_value_in_eur'] = pd.to_numeric(
        df_valuations['market_value_in_eur'], errors='coerce'
    )
    resumen_carga = pd.DataFrame([
        {'Dataset': 'players', 'Filas': len(df_players), 'Columnas cargadas': df_players.shape[1]},
        {'Dataset': 'appearances', 'Filas': len(df_appearances), 'Columnas cargadas': df_appearances.shape[1]},
        {'Dataset': 'games', 'Filas': len(df_games), 'Columnas cargadas': df_games.shape[1]},
        {'Dataset': 'player_valuations', 'Filas': len(df_valuations), 'Columnas cargadas': df_valuations.shape[1]},
    ])
    print(resumen_carga.to_string(index=False))
    print('Fechas nulas en valoraciones:', df_valuations['date'].isna().sum())
    print('Valores de mercado nulos:', df_valuations['market_value_in_eur'].isna().sum())
else:
    # Variables de preparación no necesarias en el modo caché.
    print('Filas de DF2 para regresión:', len(df2_regression))
    print('Jugadores:', df2_regression['player_id'].nunique())


**Modo de ejecución rápida:** la primera ejecución prepara los datos desde los CSV limpios y crea un caché en Drive. Si los CSV no han cambiado, las siguientes ejecuciones reutilizan `df2_regression` y saltan las secciones de preparación pesada (2–7), manteniendo la parte de modelos, evaluación y artefactos.

## 2. Comprobar claves y duplicados

Revisamos las claves antes de hacer uniones. Si una clave que debería ser única tiene duplicados, detenemos el notebook para no multiplicar filas accidentalmente.

In [ ]:
if not CACHE_CARGADO:
    # ==========================================================
    # 2. CONTROL DE CLAVES Y DUPLICADOS
    # ==========================================================

    duplicados_players = df_players.duplicated('player_id').sum()
    duplicados_appearances = df_appearances.duplicated('appearance_id').sum()
    duplicados_games = df_games.duplicated('game_id').sum()
    duplicados_valuations = df_valuations.duplicated(['player_id', 'date']).sum()

    print('Duplicados player_id en players:', duplicados_players)
    print('Duplicados appearance_id en appearances:', duplicados_appearances)
    print('Duplicados game_id en games:', duplicados_games)
    print('Duplicados jugador-fecha en valoraciones:', duplicados_valuations)

    # Duplicados en estas claves pueden provocar multiplicación de filas en los merges
    assert duplicados_players == 0, 'Hay player_id duplicados en players.csv'
    assert duplicados_appearances == 0, 'Hay appearance_id duplicados en appearances.csv'
    assert duplicados_games == 0, 'Hay game_id duplicados en games.csv'
    assert duplicados_valuations == 0, 'Hay valoraciones duplicadas por jugador y fecha'

    # Conteo de nulos en las columnas cargadas
    nulos_principales = pd.DataFrame({
        'Dataset': ['players', 'appearances', 'games', 'player_valuations'],
        'Nulos en columnas cargadas': [
            df_players.isna().sum().sum(),
            df_appearances.isna().sum().sum(),
            df_games.isna().sum().sum(),
            df_valuations.isna().sum().sum()
        ]
    })
    print('\nNulos totales en las columnas cargadas:')
    print(nulos_principales.to_string(index=False))

## 3. Exploración inicial del valor de mercado

Miramos la evolución anual y la distribución del objetivo. Como los valores son muy desiguales, usamos escala logarítmica en el histograma. El objetivo se mantiene en euros; las métricas del modelo se mostrarán en millones de euros.

In [ ]:
if not CACHE_CARGADO:
    # ==========================================================
    # 3. EXPLORACIÓN DEL OBJETIVO
    # ==========================================================

    # Resumen por año: número de valoraciones, jugadores, mediana y media
    df_val_temporal = df_valuations.dropna(subset=['date']).copy()
    df_val_temporal['valuation_year'] = df_val_temporal['date'].dt.year

    df_resumen_anual = (
        df_val_temporal.groupby('valuation_year')
        .agg(
            valoraciones=('player_id', 'size'),
            jugadores=('player_id', 'nunique'),
            mediana_eur=('market_value_in_eur', 'median'),
            media_eur=('market_value_in_eur', 'mean')
        )
        .reset_index()
    )
    df_resumen_anual['mediana_millones'] = df_resumen_anual['mediana_eur'] / 1_000_000
    df_resumen_anual['media_millones'] = df_resumen_anual['media_eur'] / 1_000_000

    print('Período de valoraciones:', df_val_temporal['date'].min(), 'a', df_val_temporal['date'].max())
    print(df_resumen_anual[['valuation_year', 'valoraciones', 'jugadores', 'mediana_millones', 'media_millones']].round(2).to_string(index=False))

    # Mediana anual: es menos sensible a valores extremos que la media
    plt.figure(figsize=(11, 4))
    plt.plot(df_resumen_anual['valuation_year'], df_resumen_anual['mediana_millones'], marker='o')
    plt.title('Mediana del valor de mercado por año')
    plt.xlabel('Año')
    plt.ylabel('Millones de euros')
    plt.xticks(df_resumen_anual['valuation_year'], rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Histograma de valores positivos en escala logarítmica
    valores_positivos = df_valuations.loc[
        df_valuations['market_value_in_eur'] > 0, 'market_value_in_eur'
    ] / 1_000_000
    print('\nValoraciones totales:', df_valuations['market_value_in_eur'].count())
    print('Valoraciones positivas:', len(valores_positivos))
    print('Mediana (M€):', round(valores_positivos.median(), 3))
    print('Media (M€):', round(valores_positivos.mean(), 3))
    print('Nulos:', df_valuations['market_value_in_eur'].isna().sum())
    print('Ceros:', (df_valuations['market_value_in_eur'] == 0).sum())
    print('Negativos:', (df_valuations['market_value_in_eur'] < 0).sum())

    if len(valores_positivos) > 0:
        bins = np.logspace(np.log10(valores_positivos.min()), np.log10(valores_positivos.max()), 45)
        plt.figure(figsize=(10, 4))
        plt.hist(valores_positivos, bins=bins)
        plt.xscale('log')
        plt.title('Distribución del valor de mercado')
        plt.xlabel('Millones de euros (escala logarítmica)')
        plt.ylabel('Valoraciones')
        plt.grid(axis='y')
        plt.tight_layout()
        plt.show()
    else:
        print('No hay valores positivos para representar.')

**Decisión inicial:** usamos el valor en euros original y no aplicamos ajuste de inflación. `valuation_year` se probará como variable temporal, pero no equivale a una corrección de inflación. Transformar el objetivo a logaritmos queda para un experimento posterior.

## 4. Calendario de temporadas

Calculamos el inicio y final real de cada competición-temporada a partir de las fechas de los partidos. Usaremos este calendario para no añadir estadísticas de temporadas que todavía no habían terminado cuando se valoró al jugador.

In [ ]:
if not CACHE_CARGADO:
    # ==========================================================
    # 4. CALENDARIO COMPETICIÓN-TEMPORADA
    # ==========================================================

    df_season_calendar = (
        df_games.dropna(subset=['date', 'competition_id', 'season'])
        .groupby(['competition_id', 'season'])
        .agg(
            season_start=('date', 'min'),
            season_end=('date', 'max'),
            partidos=('game_id', 'nunique')
        )
        .reset_index()
    )

    # Mostramos las últimas dos temporadas de algunas de las competiciones con más partidos
    top_competiciones = df_games['competition_id'].value_counts().head(8).index
    muestra_calendario = (
        df_season_calendar[df_season_calendar['competition_id'].isin(top_competiciones)]
        .sort_values(['competition_id', 'season_end'])
        .groupby('competition_id')
        .tail(2)
        .sort_values(['competition_id', 'season_end'])
    )
    print('Competiciones distintas:', df_games['competition_id'].nunique())
    print('Competición-temporadas:', len(df_season_calendar))
    print(muestra_calendario.to_string(index=False))
    print('Valoraciones con valor de mercado cero:', (df_valuations['market_value_in_eur'] == 0).sum())

## 5. Estadísticas por jugador, competición y temporada

Unimos las apariciones con los partidos para recuperar competición y temporada. Agregamos minutos, goles y asistencias por jugador/competición/temporada. Calculamos ratios por 90 más adelante, después de seleccionar las temporadas válidas.

In [ ]:
if not CACHE_CARGADO:
    # ==========================================================
    # 5. AGREGAR ESTADÍSTICAS POR JUGADOR-COMPETICIÓN-TEMPORADA
    # ==========================================================

    # Añadimos competición, temporada y fecha del partido a cada aparición
    df_appearances_season = df_appearances.merge(
        df_games[['game_id', 'competition_id', 'season', 'date']],
        on='game_id',
        how='left',
        validate='many_to_one',
        suffixes=('_appearance', '_game')
    )


    # Controles: apariciones sin temporada y discordancias de competición
    sin_partido = df_appearances_season['season'].isna().sum()
    comp_aparicion = df_appearances_season['competition_id_appearance'].astype('string')
    comp_partido = df_appearances_season['competition_id_game'].astype('string')
    discordancias_competicion = (
        comp_aparicion.notna() & comp_partido.notna() & comp_aparicion.ne(comp_partido)
    ).sum()
    print('Apariciones sin partido/temporada:', sin_partido)
    print('Discordancias de competición:', discordancias_competicion)

    # Usamos la competición del partido como referencia
    df_appearances_season['competition_id'] = df_appearances_season['competition_id_game']

    # Sumamos el rendimiento del jugador en cada competición-temporada
    df_stats_comp_season = (
        df_appearances_season.dropna(subset=['season', 'competition_id'])
        .groupby(['player_id', 'competition_id', 'season'])
        .agg(
            partidos=('appearance_id', 'nunique'),
            minutos=('minutes_played', 'sum'),
            goles=('goals', 'sum'),
            asistencias=('assists', 'sum')
        )
        .reset_index()
        .merge(
            df_season_calendar[['competition_id', 'season', 'season_end']],
            on=['competition_id', 'season'],
            how='left',
            validate='many_to_one'
        )
    )
    print('\nFilas jugador-comp.-temporada:', len(df_stats_comp_season))
    print('Jugadores incluidos:', df_stats_comp_season['player_id'].nunique())
    print('Registros sin fecha fin de temporada:', df_stats_comp_season['season_end'].isna().sum())
    print(df_stats_comp_season.head(5).round(2).to_string(index=False))

    # Liberamos la unión grande de apariciones para ahorrar memoria
    del df_appearances_season

## 6. Construir el dataset histórico `df2`

Creamos una fila por valoración positiva desde 2015. Para cada jugador y valoración buscamos, en cada competición, la última temporada terminada **antes** de la valoración. Usamos `merge_asof` para buscar la temporada anterior más cercana sin crear una unión gigante con todas las temporadas históricas.

Sumamos estadísticas de las competiciones seleccionadas y calculamos goles y asistencias por 90 minutos. La edad se calcula exactamente en la fecha de valoración.

In [ ]:
if not CACHE_CARGADO:
    # ==========================================================
    # 6. CONSTRUIR DF2 HISTÓRICO (UNA FILA POR VALORACIÓN)
    # ==========================================================

    # Base de valoraciones: positivas, con fecha y desde 2015
    df2_base = df_valuations.loc[
        (df_valuations['market_value_in_eur'] > 0)
        & (df_valuations['date'].notna())
        & (df_valuations['date'].dt.year >= 2015)
    ].copy()
    df2_base = df2_base.rename(columns={'date': 'valuation_date'}).reset_index(drop=True)
    df2_base['valuation_year'] = df2_base['valuation_date'].dt.year
    df2_base['valuation_id'] = np.arange(len(df2_base))

    # Añadimos fecha de nacimiento y posiciones desde players
    df2_base = df2_base.merge(
        df_players[['player_id', 'date_of_birth', 'position', 'sub_position']],
        on='player_id',
        how='left',
        validate='many_to_one'
    )
    df2_base['date_of_birth'] = pd.to_datetime(df2_base['date_of_birth'], errors='coerce')

    # Edad completa en la fecha de valoración (si no hay nacimiento, queda nula)
    val_date = df2_base['valuation_date']
    dob = df2_base['date_of_birth']
    birthday_not_reached = (
        (val_date.dt.month < dob.dt.month)
        | ((val_date.dt.month == dob.dt.month) & (val_date.dt.day < dob.dt.day))
    )
    df2_base['age'] = (
        val_date.dt.year - dob.dt.year - birthday_not_reached.astype('Int64')
    ).astype('Int64')

    # Pares jugador-competición en los que existen estadísticas
    # Expandimos cada valoración solo por las competiciones de ese jugador
    # (no por todas las temporadas históricas)
    df_player_competitions = (
        df_stats_comp_season.dropna(subset=['season_end'])[['player_id', 'competition_id']]
        .drop_duplicates()
    )
    df_val_comp = df2_base[['valuation_id', 'player_id', 'valuation_date']].merge(
        df_player_competitions,
        on='player_id',
        how='inner'
    )

    # Preparamos las estadísticas para buscar la temporada completada más reciente
    # Si dos temporadas tienen la misma fecha de fin para un jugador/competición, conservamos una
    df_stats_asof = (
        df_stats_comp_season.dropna(subset=['season_end'])
        .sort_values(['season_end', 'player_id', 'competition_id', 'season'])
        .drop_duplicates(['player_id', 'competition_id', 'season_end'], keep='last')
    )
    df_val_comp = df_val_comp.sort_values(['valuation_date', 'player_id', 'competition_id']).reset_index(drop=True)
    df_stats_asof = df_stats_asof.sort_values(['season_end', 'player_id', 'competition_id']).reset_index(drop=True)

    # Seleccionamos la temporada más reciente que terminó antes de cada valoración
    # allow_exact_matches=False excluye temporadas que terminan el mismo día de la valoración
    df_snapshot_comp = pd.DataFrame()
    if not df_val_comp.empty and not df_stats_asof.empty:
        df_snapshot_comp = pd.merge_asof(
            df_val_comp,
            df_stats_asof,
            left_on='valuation_date',
            right_on='season_end',
            by=['player_id', 'competition_id'],
            direction='backward',
            allow_exact_matches=False
        )
        df_snapshot_comp = df_snapshot_comp.dropna(subset=['season_end'])
        df_stats_snapshot = (
            df_snapshot_comp.groupby('valuation_id')
            .agg(
                competiciones_con_stats=('competition_id', 'nunique'),
                latest_stats_end=('season_end', 'max'),
                partidos=('partidos', 'sum'),
                minutos=('minutos', 'sum'),
                goles=('goles', 'sum'),
                asistencias=('asistencias', 'sum')
            )
            .reset_index()
        )
    else:
        # Tabla vacía con tipos de datos compatibles para el merge
        df_stats_snapshot = pd.DataFrame({
            'valuation_id': pd.Series(dtype='int64'),
            'competiciones_con_stats': pd.Series(dtype='float64'),
            'latest_stats_end': pd.Series(dtype='datetime64[ns]'),
            'partidos': pd.Series(dtype='float64'),
            'minutos': pd.Series(dtype='float64'),
            'goles': pd.Series(dtype='float64'),
            'asistencias': pd.Series(dtype='float64')
        })

    # Unimos las estadísticas a cada valoración; las que no tienen estadísticas quedan nulas
    df2 = df2_base.merge(df_stats_snapshot, on='valuation_id', how='left', validate='one_to_one')

    # Ratios por 90 minutos; no se calculan si los minutos son nulos o cero
    minutos_validos = df2['minutos'].where(df2['minutos'] > 0)
    df2['goles_por90'] = df2['goles'] / minutos_validos * 90
    df2['asistencias_por90'] = df2['asistencias'] / minutos_validos * 90

    # Conservamos variables útiles para el análisis y la futura unión con GMM
    columnas_df2 = [
        'valuation_id', 'player_id', 'valuation_date', 'valuation_year',
        'age', 'position', 'sub_position', 'competiciones_con_stats', 'latest_stats_end',
        'partidos', 'minutos', 'goles', 'asistencias', 'goles_por90', 'asistencias_por90',
        'market_value_in_eur'
    ]
    df2 = df2[columnas_df2].copy()

    # Comprobamos que las estadísticas usadas sean anteriores a la valoración
    mask_stats = df2['latest_stats_end'].notna()
    assert (df2.loc[mask_stats, 'latest_stats_end'] < df2.loc[mask_stats, 'valuation_date']).all()
    assert df2.duplicated(['player_id', 'valuation_date']).sum() == 0

    print('DF2 creado')
    print('Valoraciones:', len(df2))
    print('Jugadores:', df2['player_id'].nunique())
    print('Duplicados jugador-fecha:', df2.duplicated(['player_id', 'valuation_date']).sum())
    print('Con estadísticas:', df2['minutos'].notna().sum())
    print('Sin estadísticas:', df2['minutos'].isna().sum())
    print('Edades nulas:', df2['age'].isna().sum())
    print(df2.head(5).round(3).to_string(index=False))

    # Liberamos tablas intermedias grandes
    del df_val_comp, df_stats_asof, df_snapshot_comp, df_player_competitions, df_stats_snapshot, df2_base

## 7. Cobertura y filtros del dataset de regresión

Medimos qué porcentaje de valoraciones tiene estadísticas. Después aplicamos filtros secuenciales y guardamos una auditoría con las filas eliminadas en cada paso.

Filtros: valor de mercado positivo, estadísticas con minutos mayores que cero, edad entre 15 y 45, subposición conocida y ratios por 90 válidos. **No exigimos 450 minutos**, porque ese umbral pertenece al clustering, no a esta regresión.

In [ ]:
if not CACHE_CARGADO:
    # ==========================================================
    # 7. COBERTURA Y FILTROS
    # ==========================================================

    # Cobertura de estadísticas por año
    df2['tiene_stats'] = df2['minutos'].notna()
    df_cobertura_anual = (
        df2.groupby('valuation_year')
        .agg(valoraciones=('player_id', 'size'), con_stats=('tiene_stats', 'sum'))
        .reset_index()
    )
    df_cobertura_anual['porcentaje_con_stats'] = (
        df_cobertura_anual['con_stats'] / df_cobertura_anual['valoraciones'] * 100
    )
    print('Cobertura total de estadísticas:', round(df2['tiene_stats'].mean() * 100, 2), '%')
    print(df_cobertura_anual.round(2).to_string(index=False))

    # Filtros en orden, con auditoría de filas antes y después
    df2_regression = df2.copy()
    auditoria_filtros = []

    def aplicar_filtro(df, condicion, nombre):
        antes = len(df)
        df_filtrado = df.loc[condicion].copy()
        auditoria_filtros.append({
            'Filtro': nombre,
            'Antes': antes,
            'Después': len(df_filtrado),
            'Eliminadas': antes - len(df_filtrado)
        })
        return df_filtrado

    # 1. Valor de mercado positivo
    df2_regression = aplicar_filtro(
        df2_regression,
        df2_regression['market_value_in_eur'].notna() & (df2_regression['market_value_in_eur'] > 0),
        'Valor de mercado positivo'
    )
    # 2. Estadísticas disponibles y minutos positivos
    df2_regression = aplicar_filtro(
        df2_regression,
        df2_regression['minutos'].notna() & (df2_regression['minutos'] > 0),
        'Estadísticas y minutos > 0'
    )
    # 3. Edad razonable para la población de futbolistas
    df2_regression = aplicar_filtro(
        df2_regression,
        df2_regression['age'].between(15, 45),
        'Edad entre 15 y 45 años'
    )
    # 4. Subposición conocida
    df2_regression = aplicar_filtro(
        df2_regression,
        df2_regression['sub_position'].notna()
        & df2_regression['sub_position'].astype('string').str.strip().ne(''),
        'Subposición conocida'
    )
    # 5. Ratios por 90 válidos (sin nulos ni infinitos)
    ratios_validos = (
        df2_regression['goles_por90'].notna()
        & df2_regression['asistencias_por90'].notna()
        & np.isfinite(df2_regression[['goles_por90', 'asistencias_por90']]).all(axis=1)
    )
    df2_regression = aplicar_filtro(df2_regression, ratios_validos, 'Ratios por 90 válidos')

    # Mostrar auditoría y comprobar que jugador-fecha sea única
    df_auditoria_filtros = pd.DataFrame(auditoria_filtros)
    print('\nAUDITORÍA DE FILTROS')
    print(df_auditoria_filtros.to_string(index=False))
    print('\nDataset final para regresión')
    print('Filas:', len(df2_regression))
    print('Jugadores:', df2_regression['player_id'].nunique())
    print('Período:', df2_regression['valuation_date'].min().date(), 'a', df2_regression['valuation_date'].max().date())
    print('Duplicados jugador-fecha:', df2_regression.duplicated(['player_id', 'valuation_date']).sum())
    assert df2_regression.duplicated(['player_id', 'valuation_date']).sum() == 0

    # Guardamos el dataset ya preparado y los resúmenes necesarios para futuras ejecuciones.
    df2_regression.to_csv(CACHE_DF2, index=False)
    df_auditoria_filtros.to_csv(CACHE_AUDITORIA, index=False)
    df_cobertura_anual.to_csv(CACHE_COBERTURA, index=False)
    with open(CACHE_METADATA, 'w', encoding='utf-8') as f:
        json.dump({'firma_fuentes': FIRMA_FUENTES}, f, ensure_ascii=False, indent=2)
    print('\nCaché creado en Drive. En las siguientes ejecuciones se saltará esta preparación pesada.')


## 8. Distribución del objetivo después de los filtros

Revisamos la distribución del valor de mercado en el dataset que realmente entra al modelo. Sigue siendo muy desigual, por eso el histograma se muestra en escala logarítmica.

In [ ]:
# ==========================================================
# 8. DISTRIBUCIÓN FINAL DEL VALOR DE MERCADO
# ==========================================================

valores_finales_millones = df2_regression['market_value_in_eur'] / 1_000_000
print(valores_finales_millones.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3))

if len(valores_finales_millones) > 0:
    bins_finales = np.logspace(
        np.log10(valores_finales_millones.min()),
        np.log10(valores_finales_millones.max()),
        40
    )
    plt.figure(figsize=(9, 4))
    plt.hist(valores_finales_millones, bins=bins_finales)
    plt.xscale('log')
    plt.title('Valor de mercado en DF2 de regresión')
    plt.xlabel('Millones de euros (escala logarítmica)')
    plt.ylabel('Valoraciones')
    plt.grid(axis='y')
    plt.tight_layout()
    plt.show()

## 9. Separación temporal: Train, Validation y Test

Para elegir el modelo sin utilizar el test final, hacemos cuatro conjuntos por fecha:

- **Train candidatos:** hasta 2022.
- **Validation:** 2023, para comparar y elegir configuración.
- **Train final:** hasta 2023, una vez elegida configuración.
- **Test final:** 2024 en adelante, para medir el rendimiento futuro.

Un jugador puede aparecer en conjuntos diferentes en fechas distintas. Evaluamos el futuro de jugadores que pueden ser conocidos, no exclusivamente jugadores nuevos.

In [ ]:
# ==========================================================
# 9. CREAR CONJUNTOS TEMPORALES Y DEFINIR FEATURES
# ==========================================================

features_numericas = ['age', 'minutos', 'goles_por90', 'asistencias_por90']
features_categoricas = ['position', 'sub_position']
features_sin_anio = features_numericas + features_categoricas
features_con_anio = features_sin_anio + ['valuation_year']
target_col = 'market_value_in_eur'

# División cronológica
df_train_candidatos = df2_regression.loc[df2_regression['valuation_year'] <= 2022].copy()
df_validation = df2_regression.loc[df2_regression['valuation_year'] == 2023].copy()
df_train_final = df2_regression.loc[df2_regression['valuation_year'] <= 2023].copy()
df_test = df2_regression.loc[df2_regression['valuation_year'] >= 2024].copy()

# Comprobaciones de que los cuatro grupos tienen datos y no se solapan en el tiempo
assert not df_train_candidatos.empty, 'Train hasta 2022 está vacío'
assert not df_validation.empty, 'Validation 2023 está vacío'
assert not df_train_final.empty, 'Train final hasta 2023 está vacío'
assert not df_test.empty, 'Test desde 2024 está vacío'
assert df_train_candidatos['valuation_year'].max() <= 2022
assert df_validation['valuation_year'].min() == 2023 and df_validation['valuation_year'].max() == 2023
assert df_train_final['valuation_year'].max() <= 2023
assert df_test['valuation_year'].min() >= 2024
assert df_train_final['valuation_date'].max() < df_test['valuation_date'].min()

print('TRAIN candidatos:', len(df_train_candidatos), '|', df_train_candidatos['valuation_date'].min().date(), 'a', df_train_candidatos['valuation_date'].max().date())
print('VALIDATION:', len(df_validation), '|', df_validation['valuation_date'].min().date(), 'a', df_validation['valuation_date'].max().date())
print('TRAIN final:', len(df_train_final), '|', df_train_final['valuation_date'].min().date(), 'a', df_train_final['valuation_date'].max().date())
print('TEST final:', len(df_test), '|', df_test['valuation_date'].min().date(), 'a', df_test['valuation_date'].max().date())
print('\nFeatures sin año:', features_sin_anio)
print('Features con año:', features_con_anio)
print('Objetivo:', target_col)

## 10. Preprocesamiento y modelos

El preprocesamiento se ajusta dentro de cada pipeline, solo con Train, para evitar fuga de información:

- Variables numéricas: imputación por mediana; Ridge también escala estas columnas.
- Variables categóricas: imputación por valor más frecuente y One-Hot Encoding.
- Modelos: baseline de media, Ridge y Random Forest.
- Comparamos modelos sin año y con `valuation_year`.

Es una comparación inicial, no una búsqueda exhaustiva de hiperparámetros.

**Nota de rendimiento:** Random Forest se configura en versión ligera (25 árboles, profundidad 12 y muestreo del 70%) para que la comparación se ejecute más rápido en Colab. Es una configuración inicial, no una optimización exhaustiva.


In [ ]:
# ==========================================================
# 10. PREPROCESAMIENTO Y DEFINICIÓN DE MODELOS
# ==========================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow No se instala ni inicia servidor desde Colab.
MLFLOW_AVAILABLE = False
try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    mlflow.set_tracking_uri(MLFLOW_DIR.as_uri())
    mlflow.set_experiment('OFFBALL_Regresion_Valor_Mercado')
    print('MLflow disponible: tracking local en', mlflow.get_tracking_uri())
except ModuleNotFoundError:
    print('MLflow no está disponible. Se entrenará y guardará localmente, sin tracking de runs.')
except Exception as error:
    MLFLOW_AVAILABLE = False
    print('No se pudo activar MLflow:', type(error).__name__, '| Se continúa sin tracking.')

# Límites de entrenamiento para que la ejecución sea manejable en Colab.
RF_VALIDATION_MAX_ROWS = 60000  # muestra común para comparar todos los modelos
RF_FINAL_MAX_ROWS = 100000      # tope de entrenamiento del RF final
RF_ESTIMATORS = 20
RF_MAX_DEPTH = 10
RF_MIN_LEAF = 15
RF_MAX_FEATURES = 0.5
RF_MAX_SAMPLES = 0.5


def crear_preprocesador(features, escalar_numericas=False):
    columnas_num = [c for c in features if c in features_numericas or c == 'valuation_year']
    columnas_cat = [c for c in features if c in features_categoricas]
    transformadores = []
    if columnas_num:
        pasos_num = [('imputar', SimpleImputer(strategy='median'))]
        if escalar_numericas:
            pasos_num.append(('escalar', StandardScaler()))
        transformadores.append(('numericas', Pipeline(steps=pasos_num), columnas_num))
    if columnas_cat:
        transformadores.append((
            'categoricas',
            Pipeline(steps=[
                ('imputar', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]),
            columnas_cat
        ))
    return ColumnTransformer(transformers=transformadores, remainder='drop')


def crear_modelo(tipo, features):
    if tipo == 'dummy':
        return DummyRegressor(strategy='mean')
    if tipo == 'ridge':
        return Pipeline(steps=[
            ('preprocesado', crear_preprocesador(features, escalar_numericas=True)),
            ('modelo', Ridge(alpha=1.0))
        ])
    if tipo == 'rf':
        # RF pequeño y acotado: baseline rápida, no búsqueda exhaustiva.
        return Pipeline(steps=[
            ('preprocesado', crear_preprocesador(features, escalar_numericas=False)),
            ('modelo', RandomForestRegressor(
                n_estimators=RF_ESTIMATORS,
                max_depth=RF_MAX_DEPTH,
                min_samples_leaf=RF_MIN_LEAF,
                max_features=RF_MAX_FEATURES,
                max_samples=RF_MAX_SAMPLES,
                random_state=42,
                n_jobs=-1
            ))
        ])
    raise ValueError(f'Tipo de modelo desconocido: {tipo}')



configuraciones = [
    ('Dummy media', 'dummy', []),
    ('Ridge sin año', 'ridge', features_sin_anio),
    ('Ridge con año', 'ridge', features_con_anio),
    ('Random Forest sin año', 'rf', features_sin_anio),
    ('Random Forest con año', 'rf', features_con_anio)
]


def calcular_metricas(y_real, y_pred):
    return {
        'MAE_MEUR': mean_absolute_error(y_real, y_pred) / 1_000_000,
        'RMSE_MEUR': np.sqrt(mean_squared_error(y_real, y_pred)) / 1_000_000,
        'R2': r2_score(y_real, y_pred)
    }

print('Modelos preparados.')
print(f'Validation usa una muestra común de hasta {RF_VALIDATION_MAX_ROWS:,} filas.')
print(f'Random Forest: {RF_ESTIMATORS} árboles, profundidad {RF_MAX_DEPTH}, hoja mínima {RF_MIN_LEAF}, max_samples={RF_MAX_SAMPLES}.')
print(f'RF final: máximo {RF_FINAL_MAX_ROWS:,} filas; paralelizado con todos los cores disponibles.')
print('MLflow activo:', MLFLOW_AVAILABLE)


## 11. Entrenar y comparar en Validation

Entrenamos las cinco configuraciones con una muestra reproducible de hasta 60.000 filas del Train (hasta 2022), común para todos, y las evaluamos sobre todo Validation (2023). Si MLflow está disponible, registramos parámetros, métricas de validación y modelo. Si no, la celda sigue entrenando y muestra la comparación.

In [ ]:
# ==========================================================
# 11. ENTRENAMIENTO Y COMPARACIÓN EN VALIDATION
# ==========================================================

from contextlib import nullcontext

resultados_validacion = []
# Muestra reproducible y común para que los cinco modelos se comparen en condiciones equivalentes.
df_train_validacion = df_train_candidatos.sample(
    n=min(RF_VALIDATION_MAX_ROWS, len(df_train_candidatos)),
    random_state=42
).copy()
print(f'Train de Validation: {len(df_train_validacion):,} de {len(df_train_candidatos):,} filas (muestra común).')

for nombre, tipo, features in configuraciones:
    print(f'\nEntrenando: {nombre}...')
    modelo = crear_modelo(tipo, features)
    if tipo == 'dummy':
        X_train = np.zeros((len(df_train_validacion), 1))
        X_val = np.zeros((len(df_validation), 1))
    else:
        X_train = df_train_validacion[features].copy()
        X_val = df_validation[features].copy()
        columnas_num_modelo = [c for c in features if c in features_numericas or c == 'valuation_year']
        for conjunto in [X_train, X_val]:
            conjunto[columnas_num_modelo] = conjunto[columnas_num_modelo].apply(pd.to_numeric, errors='coerce')

    contexto_run = mlflow.start_run(run_name=nombre) if MLFLOW_AVAILABLE else nullcontext()
    with contexto_run:
        modelo.fit(X_train, df_train_validacion[target_col])
        pred_val = modelo.predict(X_val)
        metricas = calcular_metricas(df_validation[target_col], pred_val)
        run_id = mlflow.active_run().info.run_id if MLFLOW_AVAILABLE else None
        if MLFLOW_AVAILABLE:
            # Solo params y métricas en Validation; no se guardan modelos candidatos pesados.
            mlflow.log_params({
                'tipo_modelo': tipo,
                'features': ', '.join(features) if features else 'baseline media',
                'train_hasta': 2022,
                'validation_year': 2023,
                'target': target_col,
                'validation_train_rows': len(df_train_validacion),
                'alpha': 1.0 if tipo == 'ridge' else 'n/a',
                'n_estimators': RF_ESTIMATORS if tipo == 'rf' else 'n/a',
                'max_depth': RF_MAX_DEPTH if tipo == 'rf' else 'n/a',
                'max_samples': RF_MAX_SAMPLES if tipo == 'rf' else 'n/a'
            })
            mlflow.log_metrics({
                'validation_MAE_MEUR': metricas['MAE_MEUR'],
                'validation_RMSE_MEUR': metricas['RMSE_MEUR'],
                'validation_R2': metricas['R2']
            })

    resultados_validacion.append({
        'Modelo': nombre,
        'Tipo': tipo,
        'Features': ', '.join(features) if features else 'baseline media',
        'MAE_MEUR': metricas['MAE_MEUR'],
        'RMSE_MEUR': metricas['RMSE_MEUR'],
        'R2': metricas['R2'],
        'MLflow_run_id': run_id
    })
    print(f'{nombre}: MAE={metricas["MAE_MEUR"]:.3f} M€, RMSE={metricas["RMSE_MEUR"]:.3f} M€, R²={metricas["R2"]:.3f}')

# MAE más bajo es criterio principal; RMSE desempata.
df_resultados_validacion = (
    pd.DataFrame(resultados_validacion)
    .sort_values(['MAE_MEUR', 'RMSE_MEUR'])
    .reset_index(drop=True)
)
print('\nRESULTADOS DE VALIDACIÓN (2023)')
print(df_resultados_validacion[['Modelo', 'MAE_MEUR', 'RMSE_MEUR', 'R2']].round(3).to_string(index=False))


## 12. Elegir configuración, reentrenar y evaluar Test

Elegimos la configuración con menor MAE en Validation. Luego la reentrenamos hasta 2023: Ridge/Dummy usan todas las filas y Random Forest usa una muestra reproducible de hasta 100.000 filas para acotar tiempo. La evaluamos en 2024 en adelante. **Test no se usa para seleccionar el modelo.** Si MLflow está activo, registramos este modelo final y sus métricas.

In [ ]:
# ==========================================================
# 12. SELECCIÓN, REENTRENAMIENTO FINAL Y TEST
# ==========================================================

mejor_nombre = df_resultados_validacion.iloc[0]['Modelo']
mejor_configuracion = next(config for config in configuraciones if config[0] == mejor_nombre)
_, mejor_tipo, mejores_features = mejor_configuracion
print('Configuración elegida por Validation:', mejor_nombre)
print('Variables:', mejores_features if mejores_features else 'baseline de media')

# Ridge y Dummy aprovechan todas las filas. RF se limita a una muestra reproducible.
if mejor_tipo == 'rf':
    df_train_modelo_final = df_train_final.sample(
        n=min(RF_FINAL_MAX_ROWS, len(df_train_final)),
        random_state=42
    ).copy()
else:
    df_train_modelo_final = df_train_final.copy()
print(f'Train final utilizado: {len(df_train_modelo_final):,} de {len(df_train_final):,} filas.')

mejor_modelo = crear_modelo(mejor_tipo, mejores_features)
if mejor_tipo == 'dummy':
    X_train_final = np.zeros((len(df_train_modelo_final), 1))
    X_test_final = np.zeros((len(df_test), 1))
else:
    X_train_final = df_train_modelo_final[mejores_features].copy()
    X_test_final = df_test[mejores_features].copy()
    columnas_num_final = [c for c in mejores_features if c in features_numericas or c == 'valuation_year'
    ]
    for conjunto in [X_train_final, X_test_final]:
        conjunto[columnas_num_final] = conjunto[columnas_num_final].apply(pd.to_numeric, errors='coerce')

print('Reentrenando modelo final...')
mejor_modelo.fit(X_train_final, df_train_modelo_final[target_col])
print('Modelo final entrenado. Prediciendo Test (2024 en adelante)...')
pred_test = mejor_modelo.predict(X_test_final)
metricas_test = calcular_metricas(df_test[target_col], pred_test)

df_resultado_test = pd.DataFrame([{
    'Modelo_seleccionado': mejor_nombre,
    'MAE_MEUR': metricas_test['MAE_MEUR'],
    'RMSE_MEUR': metricas_test['RMSE_MEUR'],
    'R2': metricas_test['R2'],
    'Test_desde': 2024
}])
print('\nRESULTADOS TEST FINAL (2024 EN ADELANTE)')
print(df_resultado_test.round(3).to_string(index=False))

final_run_id = None
final_model_logged = False
if MLFLOW_AVAILABLE:
    with mlflow.start_run(run_name=f'FINAL - {mejor_nombre}'):
        mlflow.log_params({
            'modelo_seleccionado': mejor_nombre,
            'tipo_modelo': mejor_tipo,
            'features': ', '.join(mejores_features) if mejores_features else 'baseline media',
            'train_final_hasta': 2023,
            'validation_year': 2023,
            'test_desde': 2024,
            'target': target_col,
            'train_final_rows': len(df_train_modelo_final),
            'train_final_total_rows': len(df_train_final),
            'rf_max_rows': RF_FINAL_MAX_ROWS
        })
        mlflow.log_metrics({
            'validation_MAE_MEUR': float(df_resultados_validacion.iloc[0]['MAE_MEUR']),
            'test_MAE_MEUR': float(metricas_test['MAE_MEUR']),
            'test_RMSE_MEUR': float(metricas_test['RMSE_MEUR']),
            'test_R2': float(metricas_test['R2'])
        })
        try:
            print('Registrando el modelo final en MLflow...')
            mlflow.sklearn.log_model(mejor_modelo, artifact_path='modelo')
            final_model_logged = True
        except Exception as error:
            print('Aviso: el modelo final no pudo guardarse en MLflow:', type(error).__name__)
        final_run_id = mlflow.active_run().info.run_id

print('Run final MLflow:', final_run_id if final_run_id else 'No disponible')
print('Modelo registrado en MLflow:', final_model_logged)


## 13. Analizar errores del Test

Calculamos el error absoluto por año y mostramos valores reales frente a predicciones. El gráfico usa logaritmos solo para visualizar mejor el rango; las métricas se calcularon con las predicciones originales, sin recortarlas.

In [ ]:
# ==========================================================
# 13. ANÁLISIS DE ERRORES DEL TEST
# ==========================================================

# Tabla de predicciones y errores en euros
df_predicciones_test = df_test[
    ['valuation_id', 'player_id', 'valuation_date', 'valuation_year', 'market_value_in_eur']
].copy()
df_predicciones_test = df_predicciones_test.rename(columns={'market_value_in_eur': 'valor_real_eur'})
df_predicciones_test['prediccion_eur'] = pred_test
df_predicciones_test['error_eur'] = df_predicciones_test['prediccion_eur'] - df_predicciones_test['valor_real_eur']
df_predicciones_test['error_abs_eur'] = df_predicciones_test['error_eur'].abs()

# Resumen por año de Test
df_error_por_anio = (
    df_predicciones_test.groupby('valuation_year')
    .agg(
        valoraciones=('player_id', 'size'),
        MAE_eur=('error_abs_eur', 'mean'),
        mediana_real_eur=('valor_real_eur', 'median'),
        mediana_predicha_eur=('prediccion_eur', 'median')
    )
    .reset_index()
)
df_error_por_anio['MAE_MEUR'] = df_error_por_anio['MAE_eur'] / 1_000_000
print(df_error_por_anio.round(3).to_string(index=False))

# MAE anual: permite observar si el error aumenta en años más recientes
plt.figure(figsize=(9, 4))
plt.plot(df_error_por_anio['valuation_year'], df_error_por_anio['MAE_MEUR'], marker='o')
plt.title('MAE por año — Test final')
plt.xlabel('Año de valoración')
plt.ylabel('MAE (millones de euros)')
plt.grid(True)
plt.tight_layout()
plt.show()

# Real y predicho en log10, únicamente para el gráfico
real_log = np.log10(np.maximum(df_predicciones_test['valor_real_eur'].to_numpy(), 0) + 1)
pred_log = np.log10(np.maximum(df_predicciones_test['prediccion_eur'].to_numpy(), 0) + 1)
plt.figure(figsize=(6, 6))
plt.scatter(real_log, pred_log, alpha=0.15, s=8)
limite = max(real_log.max(), pred_log.max())
plt.plot([0, limite], [0, limite], linestyle='--')
plt.title('Valor real vs predicho (log10) — Test')
plt.xlabel('log10(valor real en € + 1)')
plt.ylabel('log10(predicción en € + 1)')
plt.grid(True)
plt.tight_layout()
plt.show()

print('Nota: predicciones negativas se muestran como cero solo en este gráfico logarítmico; no se modifican para calcular métricas.')

## 14. Consultar runs de MLflow

Usamos la librería MLflow para consultar el tracking local. No lanzamos `mlflow ui` ni ningún servidor desde el notebook. Si la librería no estaba disponible, se muestra un aviso y se pueden revisar los CSV de resultados.

In [ ]:
# ==========================================================
# 14. CONSULTA DE EXPERIMENTOS MLFLOW
# ==========================================================

if MLFLOW_AVAILABLE:
    experimento = mlflow.get_experiment_by_name('OFFBALL_Regresion_Valor_Mercado')
    if experimento is not None:
        df_runs_mlflow = mlflow.search_runs(
            experiment_ids=[experimento.experiment_id],
            order_by=['metrics.validation_MAE_MEUR ASC']
        )
        columnas_runs = [c for c in [
            'run_id', 'tags.mlflow.runName', 'metrics.validation_MAE_MEUR',
            'metrics.validation_RMSE_MEUR', 'metrics.validation_R2', 'metrics.test_MAE_MEUR'
        ] if c in df_runs_mlflow.columns]
        print(df_runs_mlflow[columnas_runs].head(10).to_string(index=False))
    else:
        print('El experimento todavía no existe.')
else:
    print('MLflow no está disponible. Los resultados se revisan en los CSV de la siguiente sección.')

## 15. Probar la carga del modelo desde MLflow

Si el modelo final se registró correctamente, lo cargamos desde su `run_id` y hacemos una predicción de prueba. Esto verifica la forma de recuperar el artefacto desde la librería MLflow; no inicia el servidor ni entrena de nuevo.


In [ ]:
# ==========================================================
# 15. CARGAR EL MODELO FINAL DESDE MLFLOW (COMPROBACIÓN OPCIONAL)
# ==========================================================

# Esta comprobación carga el artefacto desde MLflow y no es necesaria en cada ejecución.
# Activarla (True) una vez cuando queramos verificar que se puede recuperar el modelo.
COMPROBAR_CARGA_MLFLOW = False

model_uri_mlflow = f'runs:/{final_run_id}/modelo' if (final_run_id and final_model_logged) else None
if COMPROBAR_CARGA_MLFLOW and MLFLOW_AVAILABLE and model_uri_mlflow:
    try:
        print('Cargando artefacto final desde MLflow...')
        modelo_cargado_mlflow = mlflow.sklearn.load_model(model_uri_mlflow)
        predicciones_demo = modelo_cargado_mlflow.predict(X_test_final[:5])
        print('Modelo recuperado desde MLflow:', model_uri_mlflow)
        print('Predicciones de prueba (euros):', np.round(predicciones_demo, 2))
    except Exception as error:
        print('No se pudo cargar el modelo desde MLflow:', type(error).__name__)
else:
    print('Carga desde MLflow omitida (opcional). El modelo final se guardó localmente con joblib.')


## 16. Guardar dataset, métricas y modelo

Guardamos el dataset preparado, auditorías, métricas, predicciones, el modelo final y un JSON con su configuración. Estos archivos sirven para reproducir el experimento y preparar la API. El modelo local se guarda aunque MLflow no esté disponible.

In [ ]:
# ==========================================================
# 15. GUARDAR ARTEFACTOS DE LA REGRESIÓN
# ==========================================================

import joblib
import json



# Rutas de salida
ruta_df2 = OUTPUT_DIR / 'df2_regression.csv'
ruta_auditoria = OUTPUT_DIR / 'auditoria_filtros_regresion.csv'
ruta_cobertura = OUTPUT_DIR / 'cobertura_estadisticas.csv'
ruta_metricas_validacion = OUTPUT_DIR / 'metricas_validacion_regresion.csv'
ruta_metricas_test = OUTPUT_DIR / 'metricas_test_modelo_seleccionado.csv'
ruta_predicciones_test = OUTPUT_DIR / 'predicciones_test_modelo_seleccionado.csv'
ruta_error_anual = OUTPUT_DIR / 'error_test_por_anio.csv'
ruta_diccionario = OUTPUT_DIR / 'diccionario_variables_regresion.csv'
ruta_modelo = MODELS_DIR / 'mejor_modelo.joblib'
ruta_info_modelo = MODELS_DIR / 'mejor_modelo_info.json'

# Guardamos las tablas para abrirlas fuera del notebook
df2_regression.to_csv(ruta_df2, index=False)
df_auditoria_filtros.to_csv(ruta_auditoria, index=False)
df_cobertura_anual.to_csv(ruta_cobertura, index=False)
df_resultados_validacion.to_csv(ruta_metricas_validacion, index=False)
df_resultado_test.to_csv(ruta_metricas_test, index=False)
df_predicciones_test.to_csv(ruta_predicciones_test, index=False)
df_error_por_anio.to_csv(ruta_error_anual, index=False)


# Si el run final de MLflow existe, añadimos las tablas de apoyo como artefactos
if MLFLOW_AVAILABLE and final_run_id:
    try:
        with mlflow.start_run(run_id=final_run_id):
            for archivo_artifacto in [ruta_metricas_validacion, ruta_metricas_test, ruta_predicciones_test, ruta_error_anual]:
                mlflow.log_artifact(str(archivo_artifacto))
        print('Tablas de apoyo registradas como artefactos en MLflow.')
    except Exception as error:
        print('Aviso: no se pudieron registrar las tablas en MLflow:', type(error).__name__)

# Diccionario de las variables principales
diccionario_variables = pd.DataFrame([
    ('valuation_id', 'Identificador creado en este notebook para cada valoración'),
    ('player_id', 'Identificador del jugador en la fuente'),
    ('valuation_date', 'Fecha de la valoración de mercado'),
    ('valuation_year', 'Año de la valoración'),
    ('age', 'Edad completa del jugador en la fecha de valoración'),
    ('position', 'Posición general del jugador'),
    ('sub_position', 'Subposición específica del jugador'),
    ('competiciones_con_stats', 'Número de competiciones con estadísticas disponibles'),
    ('latest_stats_end', 'Fecha final de la última temporada utilizada'),
    ('partidos', 'Partidos agregados de las temporadas seleccionadas'),
    ('minutos', 'Minutos agregados de las temporadas seleccionadas'),
    ('goles', 'Goles agregados de las temporadas seleccionadas'),
    ('asistencias', 'Asistencias agregadas de las temporadas seleccionadas'),
    ('goles_por90', 'Goles por 90 calculados sobre los totales agregados'),
    ('asistencias_por90', 'Asistencias por 90 calculadas sobre los totales agregados'),
    ('market_value_in_eur', 'Objetivo: valor de mercado en euros nominales')
], columns=['Variable', 'Descripción'])
diccionario_variables.to_csv(ruta_diccionario, index=False)

# Guardamos el modelo final localmente, independientemente de MLflow
joblib.dump(mejor_modelo, ruta_modelo)

# JSON con información para reproducir o preparar la futura API
model_uri_mlflow = f'runs:/{final_run_id}/modelo' if (final_run_id and final_model_logged) else None
info_modelo = {
    'modelo_seleccionado': mejor_nombre,
    'tipo': mejor_tipo,
    'features': mejores_features,
    'target': target_col,
    'train_final_hasta': 2023,
    'validation_year': 2023,
    'test_desde': 2024,
    'train_final_rows': int(len(df_train_modelo_final)),
    'train_final_total_rows': int(len(df_train_final)),
    'validation_train_rows': int(len(df_train_validacion)),
    'validation_MAE_MEUR': float(df_resultados_validacion.iloc[0]['MAE_MEUR']),
    'test_MAE_MEUR': float(metricas_test['MAE_MEUR']),
    'test_RMSE_MEUR': float(metricas_test['RMSE_MEUR']),
    'test_R2': float(metricas_test['R2']),
    'mlflow_tracking_uri': mlflow.get_tracking_uri() if MLFLOW_AVAILABLE else None,
    'mlflow_final_run_id': final_run_id,
    'mlflow_model_uri': model_uri_mlflow
}
with open(ruta_info_modelo, 'w', encoding='utf-8') as archivo_json:
    json.dump(info_modelo, archivo_json, ensure_ascii=False, indent=2)

print('Artefactos guardados:')
for ruta in [
    ruta_df2, ruta_auditoria, ruta_cobertura, ruta_metricas_validacion,
    ruta_metricas_test, ruta_predicciones_test, ruta_error_anual,
    ruta_diccionario, ruta_modelo, ruta_info_modelo
]:
    print('-', ruta)

## 16. Qué queda para la siguiente fase

- Interpretar los resultados reales de Validation y Test; no inventamos el ganador antes de ejecutar el notebook con los datos de Drive.
- Revisar errores por posición, edad y tramo de valor, si aporta información.
- Probar transformaciones del objetivo o hiperparámetros como experimentos separados.
- Incorporar las variables de clustering/GMM del compañero y comparar contra esta línea base usando claves y fechas compatibles.
- Para despliegue, el modelo final puede recuperarse desde MLflow (`mlflow_model_uri`) si se registró. La futura puesta en marcha de MLflow y FastAPI será fuera del notebook, con un único comando (por ejemplo, Docker Compose). La API cargará el modelo y no entrenará al arrancar.

**Limitaciones a documentar:** valores nominales sin ajuste de inflación; estadísticas generales y no específicas por posición; jugadores pueden aparecer en Train y Test en fechas distintas; y las fechas no permiten ordenar eventos del mismo día.